# Analyse complète du VAE conditionnel — Modèles pré-entraînés

Ce notebook charge les modèles sauvegardés et génère une **analyse exhaustive par classe** :
- Signal généré + zoom
- Spectrogramme
- Verdicts des 3 juges (classification)
- Matrice de confusion inter-classes (où le VAE confond)
- Heatmap PSD par classe
- t-SNE de l'espace latent

## Rappel : différence Classe vs Label original

| Terme | Signification |
|---|---|
| **Label original** | ID dans le parquet (`global_activity_id`), ex: 0, 3, 7, 42 |
| **Classe (mapped)** | Index interne PyTorch 0, 1, 2... utilisé par le modèle |

Le `label_mapping` stocké dans `metadata.pkl` fait le pont entre les deux.
Si tes labels originaux sont déjà 0, 1, 2 consécutifs, les deux coïncident — c'est normal.

In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
from sklearn.manifold import TSNE

print("Imports OK")

Imports OK


## Configuration — adapter ces chemins

In [2]:
# =========================================================
# CONFIGURATION — à adapter
# =========================================================

# Choisir le dossier de modèles
# SAVE_DIR = r"C:\...\saved_models_filtered_3_labels"
SAVE_DIR = r"D:\IMDS\MALMO\text_to_imu_generation\models\saved_models_unified_dataset"

# Noms des activités (optionnel) — si tu les connais, remplace les valeurs
# Clé = label ORIGINAL (avant mapping), valeur = nom lisible
# Exemple : {0: "Marche", 1: "Course", 2: "Vélo"}
# Si tu ne les connais pas, laisser None => affichage automatique "Label X"
ACTIVITY_NAMES = None
# ACTIVITY_NAMES = {0: "Marche", 1: "Course", 2: "Repos"}

# Fréquence d'échantillonnage (Hz)
FS = 25

# Nombre de signaux générés par classe pour l'analyse
N_SAMPLES_PER_CLASS = 50

# Device
try:
    import torch_directml
    device = torch_directml.device()
    print("DirectML device utilisé")
except ImportError:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device : {device}")

Device : cuda


## Définitions des architectures

In [3]:
# =========================================================
# ARCHITECTURES (identiques à l'entraînement)
# =========================================================

class ConditionalVAE(nn.Module):
    def __init__(self, n_classes, latent_dim=32, label_emb_dim=16):
        super().__init__()
        self.n_classes = n_classes
        self.latent_dim = latent_dim
        self.label_emb_dim = label_emb_dim
        self.label_emb = nn.Embedding(n_classes, label_emb_dim)
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=5, padding=2), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.Flatten(),
        )
        encoder_out_dim = 128 * 12
        self.fc_mu = nn.Linear(encoder_out_dim + label_emb_dim, latent_dim)
        self.fc_logvar = nn.Linear(encoder_out_dim + label_emb_dim, latent_dim)
        self.dec_input = nn.Linear(latent_dim + label_emb_dim, 128 * 13)
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(128, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.ReLU(),
            nn.Conv1d(32, 1, kernel_size=5, padding=2),
        )

    def encode(self, x, y):
        h = self.encoder(x)
        y_emb = self.label_emb(y)
        h_cond = torch.cat([h, y_emb], dim=1)
        return self.fc_mu(h_cond), self.fc_logvar(h_cond)

    def reparameterize(self, mu, logvar):
        logvar = torch.clamp(logvar, min=-10, max=10)
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z, y):
        y_emb = self.label_emb(y)
        h = self.dec_input(torch.cat([z, y_emb], dim=1))
        return self.decoder(h.view(-1, 128, 13))

    def forward(self, x, y):
        x = x.float()
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, y), mu, logvar

    def sample(self, n_samples, class_id, device):
        self.eval()
        y = torch.full((n_samples,), class_id, dtype=torch.long, device=device)
        z = torch.randn(n_samples, self.latent_dim, device=device)
        with torch.no_grad():
            return self.decode(z, y)

    def encode_to_z(self, x, y):
        """Encode vers mu (pour t-SNE)"""
        self.eval()
        with torch.no_grad():
            mu, _ = self.encode(x.float(), y)
        return mu


class DeepConvLSTM(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=5, padding=2), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=5, padding=2), nn.ReLU(),
        )
        self.lstm = nn.RNN(input_size=64, hidden_size=128, num_layers=2,
                           nonlinearity="tanh", batch_first=True)
        self.fc = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, n_classes)
        )

    def forward(self, x):
        x = x.float()
        x = self.conv(x).transpose(1, 2)
        x, _ = self.lstm(x)
        return self.fc(x[:, -1, :])


class CNNSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(64, 128), nn.ReLU(), nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x.float())


class MLPSimple(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(50, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.net(x.float())


print("Architectures définies.")

Architectures définies.


## Chargement des modèles

In [4]:
# =========================================================
# CHARGEMENT METADATA + MODÈLES
# =========================================================

with open(os.path.join(SAVE_DIR, "metadata.pkl"), "rb") as f:
    metadata = pickle.load(f)

mean       = metadata["mean"]
std        = metadata["std"]
label_mapping     = metadata["label_mapping"]      # {original_id -> mapped_id}
inv_label_mapping = metadata["inv_label_mapping"]  # {mapped_id -> original_id}
WINDOW_SIZE = metadata["window_size"]
LATENT_DIM  = metadata["latent_dim"]
LABEL_EMB_DIM = metadata["label_emb_dim"]
n_classes   = metadata["n_classes"]

print("=== Mapping des labels ===")
print(f"Nombre de classes : {n_classes}")
print("Label original -> Classe interne (mapped)")
for orig, mapped in sorted(label_mapping.items()):
    name = ACTIVITY_NAMES.get(orig, f"Activité {orig}") if ACTIVITY_NAMES else f"Label original {orig}"
    print(f"  {orig:>4} -> classe {mapped}  [{name}]")

# Noms d'affichage par classe mappée
CLASS_NAMES = {}
for orig, mapped in label_mapping.items():
    if ACTIVITY_NAMES and orig in ACTIVITY_NAMES:
        CLASS_NAMES[mapped] = f"{ACTIVITY_NAMES[orig]} (orig={orig})"
    else:
        CLASS_NAMES[mapped] = f"Classe {mapped} (orig={orig})"

=== Mapping des labels ===
Nombre de classes : 23
Label original -> Classe interne (mapped)
     0 -> classe 0  [Label original 0]
     1 -> classe 1  [Label original 1]
     2 -> classe 2  [Label original 2]
     3 -> classe 3  [Label original 3]
     4 -> classe 4  [Label original 4]
     5 -> classe 5  [Label original 5]
     6 -> classe 6  [Label original 6]
     7 -> classe 7  [Label original 7]
     8 -> classe 8  [Label original 8]
     9 -> classe 9  [Label original 9]
    10 -> classe 10  [Label original 10]
    11 -> classe 11  [Label original 11]
    12 -> classe 12  [Label original 12]
    13 -> classe 13  [Label original 13]
    14 -> classe 14  [Label original 14]
    15 -> classe 15  [Label original 15]
    16 -> classe 16  [Label original 16]
    17 -> classe 17  [Label original 17]
    18 -> classe 18  [Label original 18]
    19 -> classe 19  [Label original 19]
    20 -> classe 20  [Label original 20]
    21 -> classe 21  [Label original 21]
  9000 -> classe 22  [Labe

In [5]:
# Instanciation et chargement des poids
vae_model = ConditionalVAE(n_classes=n_classes, latent_dim=LATENT_DIM, label_emb_dim=LABEL_EMB_DIM).to(device)
judge1 = DeepConvLSTM(n_classes).to(device)
judge2 = CNNSimple(n_classes).to(device)
judge3 = MLPSimple(n_classes).to(device)

vae_model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "conditional_vae.pth"), map_location=device))
judge1.load_state_dict(torch.load(os.path.join(SAVE_DIR, "DeepConvLSTM.pth"), map_location=device))
judge2.load_state_dict(torch.load(os.path.join(SAVE_DIR, "CNNSimple.pth"), map_location=device))
judge3.load_state_dict(torch.load(os.path.join(SAVE_DIR, "MLPSimple.pth"), map_location=device))

vae_model.eval()
judge1.eval()
judge2.eval()
judge3.eval()
judges = [judge1, judge2, judge3]

print("Tous les modèles chargés avec succès.")

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy.core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([_reconstruct])` or the `torch.serialization.safe_globals([_reconstruct])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by 

## Fonctions utilitaires

In [ ]:
# =========================================================
# UTILITAIRES
# =========================================================

def denormalize(x):
    return x * (std + 1e-8) + mean


def generate_class(mapped_class_id, n_samples=50):
    """Génère n_samples signaux pour la classe interne mapped_class_id.
    Retourne un tensor (n_samples, 1, WINDOW_SIZE) sur device."""
    return vae_model.sample(n_samples, mapped_class_id, device)


def judge_predictions(gen_tensor):
    """Pour un batch de signaux générés, retourne les prédictions de chaque juge.
    Retourne: probs_mean (n_classes,), pred_mapped, pred_original, confidence"""
    results = []
    for judge in judges:
        with torch.no_grad():
            logits = judge(gen_tensor)
            probs = torch.softmax(logits, dim=1)  # (n, n_classes)
            mean_probs = probs.mean(0)             # (n_classes,)
            pred_mapped = torch.argmax(mean_probs).item()
            pred_orig = inv_label_mapping[pred_mapped]
            conf = mean_probs.max().item() * 100
        results.append({
            "name": judge.__class__.__name__,
            "probs": mean_probs.cpu().numpy(),
            "pred_mapped": pred_mapped,
            "pred_orig": pred_orig,
            "conf": conf,
        })
    return results


def analyse_gait(sig, fs=25):
    """Autocorrélation pour détecter un pas rythmique."""
    mag = sig.flatten().astype(np.float32)
    mag = mag - np.mean(mag)
    corr = np.correlate(mag, mag, mode="full")[len(mag)-1:]
    corr /= (np.max(corr) + 1e-7)
    min_lag, max_lag = int(0.4 * fs), int(1.2 * fs)
    if len(corr) > max_lag:
        peak_idx = np.argmax(corr[min_lag:max_lag]) + min_lag
        return corr[peak_idx], peak_idx / fs
    return 0.0, 1e-7


print("Utilitaires définis.")

## Analyse individuelle par classe

Une figure détaillée pour chaque classe : signal, spectre, verdicts juges.

In [ ]:
# =========================================================
# ANALYSE PAR CLASSE — une figure par classe
# =========================================================

all_judge_preds = {}  # Pour la matrice de confusion plus tard
# structure : all_judge_preds[mapped_class_id][judge_name] = pred_mapped

for mapped_id in range(n_classes):
    class_name = CLASS_NAMES[mapped_id]
    print(f"\n{'='*60}")
    print(f"Génération : {class_name}")
    print(f"{'='*60}")

    # --- Génération ---
    gen_tensor = generate_class(mapped_id, n_samples=N_SAMPLES_PER_CLASS)
    gen_np_norm = gen_tensor.cpu().numpy()          # (N, 1, 50) normalisé
    gen_np = denormalize(gen_np_norm)               # dénormalisé
    full_sig = gen_np.flatten()

    # --- Analyse gait ---
    gait_score, step_t = analyse_gait(full_sig, fs=FS)
    cadence = 60 / step_t if step_t > 0 else 0

    # --- Spectrogramme (sur signal moyen) ---
    mean_sig = gen_np[:, 0, :].mean(axis=0)  # moyenne des N fenêtres
    f_welch, psd = signal.welch(full_sig, fs=FS, nperseg=min(256, len(full_sig)))
    f_spec, t_spec, Sxx = signal.spectrogram(full_sig[:min(1000, len(full_sig))], fs=FS)

    # --- Verdicts juges ---
    judge_results = judge_predictions(gen_tensor)
    all_judge_preds[mapped_id] = {r["name"]: r["pred_mapped"] for r in judge_results}

    # =========================================================
    # FIGURE
    # =========================================================
    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(f"Analyse VAE — {class_name}", fontsize=14, fontweight="bold")

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # --- 1. Signal concatené (N fenêtres) ---
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(full_sig[:min(500, len(full_sig))], color="steelblue", linewidth=0.8)
    ax1.set_title(f"Signal généré — {N_SAMPLES_PER_CLASS} fenêtres concaténées (500 premiers pts)")
    ax1.set_xlabel("Échantillon")
    ax1.set_ylabel("Accél. (dénorm.)")
    ax1.grid(True, alpha=0.3)

    # --- 2. Signal moyen (une fenêtre moyenne) ---
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(mean_sig, color="darkorange", marker=".", markersize=3)
    ax2.fill_between(
        range(WINDOW_SIZE),
        gen_np[:, 0, :].min(axis=0),
        gen_np[:, 0, :].max(axis=0),
        alpha=0.15, color="darkorange", label="min/max"
    )
    ax2.set_title("Fenêtre moyenne ± min/max")
    ax2.set_xlabel("Pas de temps")
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    # --- 3. PSD Welch ---
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.semilogy(f_welch, psd, color="purple")
    ax3.set_title("PSD Welch")
    ax3.set_xlabel("Fréquence (Hz)")
    ax3.set_ylabel("Densité")
    ax3.set_xlim([0, FS / 2])
    ax3.grid(True, alpha=0.3)

    # --- 4. Spectrogramme ---
    ax4 = fig.add_subplot(gs[1, 2])
    pcm = ax4.pcolormesh(t_spec, f_spec, 10 * np.log10(Sxx + 1e-7), shading="gouraud", cmap="viridis")
    ax4.set_title("Spectrogramme (dB)")
    ax4.set_xlabel("Temps (s)")
    ax4.set_ylabel("Fréquence (Hz)")
    plt.colorbar(pcm, ax=ax4, label="dB")

    # --- 5. Verdicts juges (barres de probabilité) ---
    ax5 = fig.add_subplot(gs[2, :])
    ax5.axis("off")

    verdict_lines = [f"VERDICTS DES JUGES (label conditionné = classe {mapped_id}, orig={inv_label_mapping[mapped_id]})\n"]

    for r in judge_results:
        is_correct = r["pred_mapped"] == mapped_id
        symbol = "✅" if is_correct else "❌"
        # Distribution sur toutes les classes
        dist_str = "  |  ".join(
            f"cls{c}={r['probs'][c]*100:.1f}%" for c in range(n_classes)
        )
        verdict_lines.append(
            f"{symbol} {r['name']:20s}: prédit classe {r['pred_mapped']} (orig={r['pred_orig']})  "
            f"conf={r['conf']:.1f}%\n"
            f"    Distribution: {dist_str}"
        )

    verdict_lines.append(f"\nGAIT score={gait_score:.3f}  |  Cadence estimée={cadence:.1f} pas/min")

    ax5.text(
        0.01, 0.95, "\n".join(verdict_lines),
        fontsize=10, family="monospace",
        verticalalignment="top", transform=ax5.transAxes,
        bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8)
    )

    plt.tight_layout()
    plt.show()

## Matrice de confusion des juges

Pour chaque classe générée (ligne), quelle classe le juge prédit (colonne) ?

**Lecture :** si la diagonale est rouge, le VAE génère bien. Si une ligne pointe vers une autre colonne, le VAE a du mal avec cette classe-là.

In [ ]:
# =========================================================
# MATRICE DE CONFUSION INTER-CLASSES (par juge)
# =========================================================
# Pour chaque classe générée, on regarde la distribution de probabilité
# moyenne sur toutes les classes => matrice (n_classes x n_classes)

# Recalcul avec plus de samples et récupération des probs complètes
N_CONF = 200  # samples par classe pour la matrice

# confusion_matrices[judge_name] = tableau (n_classes, n_classes)
# ligne i = classe générée, colonne j = probabilité moyenne que juge dise "classe j"
confusion_matrices = {j.__class__.__name__: np.zeros((n_classes, n_classes)) for j in judges}

print(f"Calcul matrice de confusion ({N_CONF} samples / classe)...")

for mapped_id in range(n_classes):
    gen_tensor = generate_class(mapped_id, n_samples=N_CONF)
    for judge in judges:
        with torch.no_grad():
            logits = judge(gen_tensor)
            probs = torch.softmax(logits, dim=1).mean(0).cpu().numpy()  # (n_classes,)
        confusion_matrices[judge.__class__.__name__][mapped_id] = probs

print("Calcul terminé.")

# Affichage
tick_labels = [CLASS_NAMES[i].split(" (")[0] for i in range(n_classes)]

fig, axes = plt.subplots(1, len(judges), figsize=(7 * len(judges), 6))
if len(judges) == 1:
    axes = [axes]

for ax, judge in zip(axes, judges):
    name = judge.__class__.__name__
    mat = confusion_matrices[name]

    im = ax.imshow(mat, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")

    ax.set_xticks(range(n_classes))
    ax.set_yticks(range(n_classes))
    ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(tick_labels, fontsize=9)
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe générée")
    ax.set_title(f"Juge : {name}")

    # Valeurs dans chaque case
    for i in range(n_classes):
        for j in range(n_classes):
            ax.text(j, i, f"{mat[i, j]:.2f}",
                    ha="center", va="center", fontsize=9,
                    color="black" if mat[i, j] < 0.7 else "white")

    plt.colorbar(im, ax=ax, label="Probabilité moyenne")

fig.suptitle(
    "Matrice de confusion VAE\n"
    "Ligne = classe générée, Colonne = classe prédite\n"
    "Diagonale verte = VAE maîtrisé, hors-diagonale = confusion",
    fontsize=12
)
plt.tight_layout()
plt.show()

## Heatmap PSD comparative entre classes

Permet de voir si chaque classe a un **profil fréquentiel distinct**. Si deux classes ont la même heatmap, le VAE ne les différencie pas spectralement.

In [ ]:
# =========================================================
# HEATMAP PSD COMPARATIVE ENTRE CLASSES
# =========================================================

psd_matrix = []  # (n_classes, n_freqs)
freq_axis = None

for mapped_id in range(n_classes):
    gen_tensor = generate_class(mapped_id, n_samples=100)
    gen_np = denormalize(gen_tensor.cpu().numpy()).reshape(-1)  # tout aplati
    f_w, psd_vals = signal.welch(gen_np, fs=FS, nperseg=256)
    psd_matrix.append(psd_vals)
    if freq_axis is None:
        freq_axis = f_w

psd_matrix = np.array(psd_matrix)  # (n_classes, n_freqs)

# Normalisation en dB
psd_db = 10 * np.log10(psd_matrix + 1e-12)

fig, ax = plt.subplots(figsize=(14, 1.5 * n_classes + 2))

im = ax.imshow(
    psd_db,
    aspect="auto",
    cmap="plasma",
    extent=[freq_axis[0], freq_axis[-1], n_classes - 0.5, -0.5]
)

ax.set_yticks(range(n_classes))
ax.set_yticklabels([CLASS_NAMES[i] for i in range(n_classes)], fontsize=10)
ax.set_xlabel("Fréquence (Hz)", fontsize=11)
ax.set_title("PSD Welch par classe (dB) — différences fréquentielles entre classes", fontsize=12)

plt.colorbar(im, ax=ax, label="PSD (dB)")
plt.tight_layout()
plt.show()

# Affichage superposé des PSD
fig2, ax2 = plt.subplots(figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, n_classes))
for mapped_id in range(n_classes):
    ax2.semilogy(freq_axis, psd_matrix[mapped_id], label=CLASS_NAMES[mapped_id], color=colors[mapped_id], linewidth=1.5)
ax2.set_xlabel("Fréquence (Hz)")
ax2.set_ylabel("PSD")
ax2.set_title("Comparaison PSD Welch — toutes les classes superposées")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([0, FS / 2])
plt.tight_layout()
plt.show()

## t-SNE de l'espace latent

Encode des échantillons générés vers `mu` et projette en 2D. Si les classes forment des **clusters séparés**, le VAE a bien appris des représentations distinctes. Si elles se mélangent, c'est là que le problème se cache.

In [ ]:
# =========================================================
# t-SNE DE L'ESPACE LATENT
# =========================================================
# On génère des signaux et on les ré-encode pour récupérer mu.
# (On échantillonne directement z ~ N(0,I) et on décode,
# puis on ré-encode pour voir où ils tombent dans l'espace latent.)

N_TSNE = 80  # par classe

all_z = []
all_labels_tsne = []

for mapped_id in range(n_classes):
    # Générer
    gen_tensor = generate_class(mapped_id, n_samples=N_TSNE)  # (N, 1, 50)
    y_tensor = torch.full((N_TSNE,), mapped_id, dtype=torch.long, device=device)

    # Ré-encoder vers mu
    mu = vae_model.encode_to_z(gen_tensor, y_tensor)
    all_z.append(mu.cpu().numpy())
    all_labels_tsne.extend([mapped_id] * N_TSNE)

all_z = np.concatenate(all_z, axis=0)           # (N*n_classes, latent_dim)
all_labels_tsne = np.array(all_labels_tsne)

print(f"t-SNE sur {len(all_z)} points (latent_dim={LATENT_DIM})...")
tsne = TSNE(n_components=2, perplexity=min(30, N_TSNE - 1), random_state=42, n_iter=1000)
z_2d = tsne.fit_transform(all_z)
print("t-SNE terminé.")

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.tab10(np.linspace(0, 1, n_classes))

for mapped_id in range(n_classes):
    mask = all_labels_tsne == mapped_id
    ax.scatter(
        z_2d[mask, 0], z_2d[mask, 1],
        label=CLASS_NAMES[mapped_id],
        color=colors[mapped_id],
        alpha=0.6, s=20
    )

ax.set_title(
    "t-SNE de l'espace latent mu (signaux générés puis ré-encodés)\n"
    "Clusters séparés = VAE maîtrisé | Clusters mélangés = confusion",
    fontsize=11
)
ax.legend(fontsize=9, markerscale=2)
ax.set_xlabel("t-SNE dim 1")
ax.set_ylabel("t-SNE dim 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## Résumé synthétique

In [ ]:
# =========================================================
# RÉSUMÉ SYNTHÉTIQUE
# =========================================================

print("\n" + "="*70)
print("RÉSUMÉ SYNTHÉTIQUE — Qualité VAE par classe")
print("="*70)
print(f"{'Classe':<35} {'DeepConvLSTM':^15} {'CNNSimple':^12} {'MLPSimple':^12}")
print("-"*70)

N_FINAL = 200
for mapped_id in range(n_classes):
    gen_tensor = generate_class(mapped_id, n_samples=N_FINAL)
    judge_results = judge_predictions(gen_tensor)

    name = CLASS_NAMES[mapped_id]
    verdicts = []
    for r in judge_results:
        is_correct = r["pred_mapped"] == mapped_id
        symbol = "✅" if is_correct else f"❌→cls{r['pred_mapped']}"
        verdicts.append(f"{symbol}({r['conf']:.0f}%)")

    print(f"{name:<35} {verdicts[0]:^15} {verdicts[1]:^12} {verdicts[2]:^12}")

print("="*70)
print("\n✅ = juge prédit la bonne classe")
print("❌→clsX = juge confond avec la classe X")
print("(conf%) = confiance moyenne du juge sur N=200 signaux générés")